# K-Fold v5 Comparative (v3 CE vs v4 KL+Jaccard)

This notebook compares Jaccard performance of uploaded v3 and v4 checkpoints on the same evaluation set using the same span-extraction helper.


In [ ]:
import os
import sys
import json
import subprocess
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:128")

REPO_URL = "https://github.com/mruniverse8/kaggle-experiments-.git"
REPO_DIR = Path("/kaggle/working/kaggle-experiments-")
BRANCH = "v5_compartive"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run(["git", "fetch", "--all"], check=True)
subprocess.run(["git", "checkout", BRANCH], check=True)
subprocess.run(["git", "reset", "--hard", f"origin/{BRANCH}"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

current_branch = subprocess.check_output(["git", "rev-parse", "--abbrev-ref", "HEAD"]).decode().strip()
current_commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"]).decode().strip()
print("Git branch:", current_branch)
print("Git commit:", current_commit)
print("Repo ready at:", REPO_DIR)


## Checkpoint Inputs

Upload v3 and v4 checkpoint files to Kaggle datasets, then update `v3_checkpoint` and `v4_checkpoint` paths in config if needed.


In [ ]:
import os
import sys
from pathlib import Path
import json
import pandas as pd

sys.path.insert(0, str(Path("src").resolve()))
from dz2_causal.compare_v3_v4_jaccard import load_compare_config, run_comparison

CFG_PATH = os.environ.get("CFG_PATH", "config/kaggle_compare_v3_v4_jaccard.json")
print("Using comparison config:", CFG_PATH)
cfg = load_compare_config(CFG_PATH)
cfg


In [ ]:
results = run_comparison(cfg)
results["summary"]


In [ ]:
out_dir = Path(cfg.output_dir)
v3_metrics = json.loads((out_dir / "v3_metrics.json").read_text())
v4_metrics = json.loads((out_dir / "v4_metrics.json").read_text())
summary = json.loads((out_dir / "comparison_summary.json").read_text())
preds_df = pd.read_csv(out_dir / "v3_vs_v4_predictions.csv")

print("v3 mean_jaccard:", v3_metrics["mean_jaccard"])
print("v4 mean_jaccard:", v4_metrics["mean_jaccard"])
print("delta v4-v3:", summary["delta_v4_minus_v3"])
print("winner_counts:", summary["winner_counts"])

preds_df.head()
